# 07c - Tuning orientado a ESI 4/5

Este notebook prueba una busqueda nocturna separada para mejorar las clases de cola, especialmente ESI 4 y ESI 5, sin usar el test temporal y sin sobrescribir el modelo final congelado.

El experimento parte de la arquitectura ganadora `lightgbm_final_bert`: variables tabulares finales + componentes BERT/SVD.

## 1. Protocolo

- Validacion: `StratifiedGroupKFold` agrupado por paciente.
- Datos: solo train y predicciones OOF existentes del modelo actual.
- Objetivo: score compuesto con Macro F1, F1 de ESI 4, F1 de ESI 5 y Balanced Accuracy.
- Penalizaciones: degradacion de F1 A1/A2 y caida de Macro F1 frente al modelo actual.
- El test temporal no se carga ni se usa.

In [1]:
# ruff: noqa: E402, I001
import json
import sys
import time
import warnings
from pathlib import Path
from typing import Any

import lightgbm as lgb
import numpy as np
import optuna
import pandas as pd
from sklearn.metrics import (
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    precision_recall_fscore_support,
)
from sklearn.model_selection import StratifiedGroupKFold

warnings.filterwarnings("ignore")

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "pyproject.toml").exists():
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise RuntimeError("No se pudo localizar la raiz del proyecto.")
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from triaje_ia.config import DATA_PROCESSED, MODELS_DIR, REPORTS_DIR

RANDOM_STATE = 42
CLASSES = np.array([1, 2, 3, 4, 5])
N_SPLITS = 5
N_TRIALS = 100
POLICY_TRIALS = 800
TIMEOUT_SECONDS = None

OUT_DIR = REPORTS_DIR / "hyperparameter_tuning"
OUT_DIR.mkdir(parents=True, exist_ok=True)

STUDY_NAME = "lgbm_bert_tail_class_tuning"
POLICY_STUDY_NAME = "lgbm_bert_tail_policy_oof"
STORAGE_URL = f"sqlite:///{(OUT_DIR / 'lgbm_bert_tail_tuning_study.db').as_posix()}"
POLICY_STORAGE_URL = f"sqlite:///{(OUT_DIR / 'lgbm_bert_tail_policy_study.db').as_posix()}"

print(f"Proyecto: {PROJECT_ROOT}")
print(f"Salida: {OUT_DIR}")

Proyecto: c:\Users\CARLOS\triaje-ia-tfg
Salida: C:\Users\CARLOS\triaje-ia-tfg\reports\hyperparameter_tuning


c:\Users\CARLOS\triaje-ia-tfg\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Carga de train y baseline OOF

Se reconstruye la matriz de entrenamiento del modelo final. El baseline se calcula desde `oof_predictions.parquet`, por lo que incluye metricas completas de las cinco clases.

In [2]:
required = [
    DATA_PROCESSED / "X_train_final.parquet",
    DATA_PROCESSED / "bert_embeddings_train.parquet",
    DATA_PROCESSED / "y_train.parquet",
    DATA_PROCESSED / "groups_train.npy",
    MODELS_DIR / "feature_list.json",
    DATA_PROCESSED / "oof_predictions.parquet",
]
missing = [path for path in required if not path.exists()]
if missing:
    raise FileNotFoundError("Faltan artefactos: " + ", ".join(map(str, missing)))

feature_payload = json.loads((MODELS_DIR / "feature_list.json").read_text(encoding="utf-8"))
feature_list = feature_payload["features"]

X_tab = pd.read_parquet(DATA_PROCESSED / "X_train_final.parquet").reset_index(drop=True)
X_bert = pd.read_parquet(DATA_PROCESSED / "bert_embeddings_train.parquet").reset_index(drop=True)
X_train = pd.concat([X_tab, X_bert], axis=1)[feature_list]
y_train = pd.read_parquet(DATA_PROCESSED / "y_train.parquet").squeeze().astype(int).reset_index(drop=True)
groups_train = np.load(DATA_PROCESSED / "groups_train.npy")

assert X_train.shape[0] == y_train.shape[0] == len(groups_train)
assert X_train.shape[1] == 79
assert y_train.isin(CLASSES).all()

oof_all = pd.read_parquet(DATA_PROCESSED / "oof_predictions.parquet")
baseline_oof = oof_all[oof_all["modelo"] == "lightgbm_final_bert"].sort_values("row_id").reset_index(drop=True)
assert len(baseline_oof) == len(y_train)
assert np.array_equal(baseline_oof["y_true"].to_numpy(), y_train.to_numpy())

print("X_train:", X_train.shape)
print("y_train:", y_train.shape)
print("Distribucion train:")
print(y_train.value_counts(normalize=True).sort_index().round(4).to_string())

X_train: (334480, 79)
y_train: (334480,)
Distribucion train:
acuity
1    0.0579
2    0.3326
3    0.5372
4    0.0695
5    0.0028


## 3. Metricas, score compuesto y pesos

El score compuesto busca mejorar A4/A5, pero penaliza degradaciones clinicamente delicadas en A1/A2.

In [3]:
def metricas_completas(y_true: np.ndarray | pd.Series, y_pred: np.ndarray) -> dict[str, float]:
    precision, recall, f1, support = precision_recall_fscore_support(
        y_true, y_pred, labels=CLASSES, zero_division=0
    )
    metrics: dict[str, float] = {
        "macro_f1": float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
        "weighted_f1": float(f1_score(y_true, y_pred, average="weighted", zero_division=0)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),
    }
    for i, cls in enumerate(CLASSES):
        metrics[f"precision_a{cls}"] = float(precision[i])
        metrics[f"recall_a{cls}"] = float(recall[i])
        metrics[f"f1_a{cls}"] = float(f1[i])
        metrics[f"support_a{cls}"] = float(support[i])

    y_true_arr = np.asarray(y_true, dtype=int)
    y_pred_arr = np.asarray(y_pred, dtype=int)
    metrics["infratriaje_total"] = float(np.mean(y_pred_arr > y_true_arr))
    metrics["sobretriaje_total"] = float(np.mean(y_pred_arr < y_true_arr))
    metrics["infratriaje_critico_a1"] = float(np.mean((y_true_arr == 1) & (y_pred_arr > 1)))
    return metrics


def score_compuesto(metrics: dict[str, float], baseline: dict[str, float]) -> float:
    score = (
        0.55 * metrics["macro_f1"]
        + 0.20 * metrics["f1_a4"]
        + 0.15 * metrics["f1_a5"]
        + 0.10 * metrics["balanced_accuracy"]
    )
    penalty = 0.0
    penalty += 4.0 * max(0.0, baseline["f1_a1"] - metrics["f1_a1"] - 0.005)
    penalty += 3.0 * max(0.0, baseline["f1_a2"] - metrics["f1_a2"] - 0.005)
    penalty += 2.0 * max(0.0, baseline["macro_f1"] - metrics["macro_f1"])
    penalty += 2.0 * max(0.0, metrics["infratriaje_critico_a1"] - baseline["infratriaje_critico_a1"] - 0.001)
    return float(score - penalty)


def balanced_base_weights(y: pd.Series) -> np.ndarray:
    counts = y.value_counts().sort_index()
    n = len(y)
    k = len(CLASSES)
    class_weight = {int(cls): n / (k * int(counts.loc[cls])) for cls in CLASSES}
    return y.map(class_weight).to_numpy(dtype=float)


BASE_BALANCED_WEIGHTS = balanced_base_weights(y_train)


def build_sample_weights(weight_power: float, tail_boost_a4: float, tail_boost_a5: float) -> np.ndarray:
    weights = np.power(BASE_BALANCED_WEIGHTS, weight_power)
    y_arr = y_train.to_numpy()
    weights[y_arr == 4] *= tail_boost_a4
    weights[y_arr == 5] *= tail_boost_a5
    weights = np.minimum(weights, 25.0)
    weights = weights / weights.mean()
    return weights


baseline_metrics = metricas_completas(y_train, baseline_oof["y_pred"].to_numpy())
baseline_score = score_compuesto(baseline_metrics, baseline_metrics)
print(json.dumps({k: round(v, 6) for k, v in baseline_metrics.items() if k in ["macro_f1", "balanced_accuracy", "f1_a1", "f1_a2", "f1_a3", "f1_a4", "f1_a5", "infratriaje_critico_a1"]}, indent=2))
print("baseline_score:", round(baseline_score, 6))

{
  "macro_f1": 0.56794,
  "balanced_accuracy": 0.586992,
  "f1_a1": 0.669482,
  "f1_a2": 0.662288,
  "f1_a3": 0.740868,
  "f1_a4": 0.500692,
  "f1_a5": 0.266368,
  "infratriaje_critico_a1": 0.018097
}
baseline_score: 0.51116


## 4. Optuna: hiperparametros + balanceo

Cada trial entrena 5 folds y guarda metricas OOF completas. La base SQLite permite reanudar el estudio durante la noche.

In [4]:
def fixed_lgbm_params() -> dict[str, Any]:
    return {
        "objective": "multiclass",
        "num_class": 5,
        "metric": "multi_logloss",
        "boosting_type": "gbdt",
        "random_state": RANDOM_STATE,
        "n_jobs": -1,
        "verbose": -1,
        "subsample_freq": 1,
    }


def suggest_params(trial: optuna.Trial) -> tuple[dict[str, Any], dict[str, float]]:
    params = fixed_lgbm_params()
    params.update(
        {
            "num_leaves": trial.suggest_int("num_leaves", 64, 180),
            "max_depth": trial.suggest_int("max_depth", 6, 11),
            "min_data_in_leaf": trial.suggest_int("min_data_in_leaf", 20, 160),
            "reg_alpha": trial.suggest_float("reg_alpha", 1e-3, 2.0, log=True),
            "reg_lambda": trial.suggest_float("reg_lambda", 1e-3, 4.0, log=True),
            "min_gain_to_split": trial.suggest_float("min_gain_to_split", 0.0, 1.2),
            "subsample": trial.suggest_float("subsample", 0.70, 0.95),
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.50, 0.85),
            "learning_rate": trial.suggest_float("learning_rate", 0.008, 0.035, log=True),
            "n_estimators": trial.suggest_int("n_estimators", 600, 1600),
        }
    )
    weight_params = {
        "weight_power": trial.suggest_float("weight_power", 0.45, 0.85),
        "tail_boost_a4": trial.suggest_float("tail_boost_a4", 1.0, 2.5),
        "tail_boost_a5": trial.suggest_float("tail_boost_a5", 1.0, 6.0),
    }
    return params, weight_params


def params_from_trial(trial: optuna.trial.FrozenTrial) -> tuple[dict[str, Any], dict[str, float]]:
    params = fixed_lgbm_params()
    params.update({k: v for k, v in trial.params.items() if not k.startswith("tail_") and k != "weight_power"})
    weight_params = {
        "weight_power": float(trial.params["weight_power"]),
        "tail_boost_a4": float(trial.params["tail_boost_a4"]),
        "tail_boost_a5": float(trial.params["tail_boost_a5"]),
    }
    return params, weight_params


def evaluate_config(params: dict[str, Any], weight_params: dict[str, float], return_oof: bool = False) -> dict[str, Any]:
    cv = StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=False)
    sample_weights = build_sample_weights(**weight_params)
    oof_pred = np.zeros(len(y_train), dtype=int)
    oof_proba = np.zeros((len(y_train), 5), dtype=float)
    fold_scores: list[float] = []
    best_iterations: list[int] = []

    for fold, (idx_tr, idx_val) in enumerate(cv.split(X_train, y_train, groups_train), start=1):
        model = lgb.LGBMClassifier(**params)
        model.fit(
            X_train.iloc[idx_tr],
            y_train.iloc[idx_tr] - 1,
            sample_weight=sample_weights[idx_tr],
            eval_set=[(X_train.iloc[idx_val], y_train.iloc[idx_val] - 1)],
            eval_sample_weight=[sample_weights[idx_val]],
            callbacks=[lgb.early_stopping(50, verbose=False)],
        )
        proba = model.predict_proba(X_train.iloc[idx_val])
        pred = np.argmax(proba, axis=1) + 1
        oof_proba[idx_val] = proba
        oof_pred[idx_val] = pred
        fold_scores.append(float(f1_score(y_train.iloc[idx_val], pred, average="macro", zero_division=0)))
        best_iterations.append(int(model.best_iteration_ or params["n_estimators"]))

    metrics = metricas_completas(y_train, oof_pred)
    metrics["score_compuesto"] = score_compuesto(metrics, baseline_metrics)
    metrics["fold_macro_f1_mean"] = float(np.mean(fold_scores))
    metrics["fold_macro_f1_std"] = float(np.std(fold_scores))
    metrics["best_iteration_mean"] = float(np.mean(best_iterations))
    metrics["best_iteration_std"] = float(np.std(best_iterations))
    metrics["sample_weight_min"] = float(sample_weights.min())
    metrics["sample_weight_max"] = float(sample_weights.max())
    metrics["sample_weight_mean"] = float(sample_weights.mean())
    result: dict[str, Any] = {"metrics": metrics}
    if return_oof:
        result["oof_pred"] = oof_pred
        result["oof_proba"] = oof_proba
    return result

In [5]:
def objective(trial: optuna.Trial) -> float:
    params, weight_params = suggest_params(trial)
    start = time.perf_counter()
    result = evaluate_config(params, weight_params, return_oof=False)
    elapsed = time.perf_counter() - start
    metrics = result["metrics"]

    for key, value in {**metrics, **weight_params}.items():
        trial.set_user_attr(key, float(value))
    trial.set_user_attr("elapsed_seconds", float(elapsed))

    return float(metrics["score_compuesto"])


sampler = optuna.samplers.TPESampler(seed=RANDOM_STATE, multivariate=True)
pruner = optuna.pruners.MedianPruner(n_startup_trials=10, n_warmup_steps=2)
study = optuna.create_study(
    study_name=STUDY_NAME,
    storage=STORAGE_URL,
    direction="maximize",
    sampler=sampler,
    pruner=pruner,
    load_if_exists=True,
)

study.optimize(objective, n_trials=N_TRIALS, timeout=TIMEOUT_SECONDS, show_progress_bar=True)

print("Best trial:", study.best_trial.number)
print("Best score:", study.best_value)
print(json.dumps(study.best_trial.params, indent=2))

[I 2026-06-06 23:50:39,865] A new study created in RDB with name: lgbm_bert_tail_class_tuning
Best trial: 0. Best value: 0.450772:   1%|          | 1/100 [03:44<6:10:18, 224.43s/it]

[I 2026-06-06 23:54:24,288] Trial 0 finished with value: 0.45077192110528963 and parameters: {'num_leaves': 107, 'max_depth': 11, 'min_data_in_leaf': 123, 'reg_alpha': 0.09466503798478172, 'reg_lambda': 0.0036474429068442276, 'min_gain_to_split': 0.18719342440344316, 'subsample': 0.7145209030420498, 'colsample_bytree': 0.8031616510212273, 'learning_rate': 0.019426363885157984, 'n_estimators': 1308, 'weight_power': 0.458233797718321, 'tail_boost_a4': 2.4548647782429915, 'tail_boost_a5': 5.162213204002109}. Best is trial 0 with value: 0.45077192110528963.


Best trial: 0. Best value: 0.450772:   2%|▏         | 2/100 [10:04<8:36:22, 316.15s/it]

[I 2026-06-07 00:00:44,649] Trial 1 finished with value: 0.40203114330364564 and parameters: {'num_leaves': 88, 'max_depth': 7, 'min_data_in_leaf': 45, 'reg_alpha': 0.010099799910483423, 'reg_lambda': 0.07766120906821304, 'min_gain_to_split': 0.5183340223705389, 'subsample': 0.7728072850495105, 'colsample_bytree': 0.7141485131528328, 'learning_rate': 0.009828845101145204, 'n_estimators': 892, 'weight_power': 0.5965447373174767, 'tail_boost_a4': 1.6841049763255538, 'tail_boost_a5': 4.925879806965068}. Best is trial 0 with value: 0.45077192110528963.


Best trial: 0. Best value: 0.450772:   3%|▎         | 3/100 [12:44<6:35:47, 244.82s/it]

...

[I 2026-06-07 09:13:47,136] Trial 97 finished with value: 0.41069275223943824 and parameters: {'num_leaves': 132, 'max_depth': 11, 'min_data_in_leaf': 52, 'reg_alpha': 0.566081548530693, 'reg_lambda': 0.053232977823862965, 'min_gain_to_split': 0.4130589357393216, 'subsample': 0.854831643603058, 'colsample_bytree': 0.6582986295191839, 'learning_rate': 0.00837869559008248, 'n_estimators': 749, 'weight_power': 0.6274993321914788, 'tail_boost_a4': 1.4145993486151132, 'tail_boost_a5': 1.6721842034106076}. Best is trial 84 with value: 0.5144535967292019.


Best trial: 84. Best value: 0.514454:  99%|█████████▉| 99/100 [9:30:40<06:53, 413.03s/it]

[I 2026-06-07 09:21:19,942] Trial 98 finished with value: 0.5067190715826931 and parameters: {'num_leaves': 145, 'max_depth': 11, 'min_data_in_leaf': 32, 'reg_alpha': 0.762310367398682, 'reg_lambda': 0.03306360125431721, 'min_gain_to_split': 0.815354877908318, 'subsample': 0.8112146499885229, 'colsample_bytree': 0.6553376002589715, 'learning_rate': 0.010244281436977845, 'n_estimators': 780, 'weight_power': 0.4909039745799671, 'tail_boost_a4': 1.1435255542399951, 'tail_boost_a5': 1.8949236794980107}. Best is trial 84 with value: 0.5144535967292019.


Best trial: 84. Best value: 0.514454: 100%|██████████| 100/100 [9:38:18<00:00, 346.98s/it]

[I 2026-06-07 09:28:58,156] Trial 99 finished with value: 0.5035975922404191 and parameters: {'num_leaves': 137, 'max_depth': 10, 'min_data_in_leaf': 49, 'reg_alpha': 0.9049766351254075, 'reg_lambda': 0.01921134283155721, 'min_gain_to_split': 0.9515361603676831, 'subsample': 0.83669542923777, 'colsample_bytree': 0.6461104964609873, 'learning_rate': 0.009987924479305127, 'n_estimators': 845, 'weight_power': 0.4640301545004495, 'tail_boost_a4': 1.4434603423699968, 'tail_boost_a5': 2.323108306846283}. Best is trial 84 with value: 0.5144535967292019.
Best trial: 84
Best score: 0.5144535967292019
{
  "num_leaves": 154,
  "max_depth": 7,
  "min_data_in_leaf": 68,
  "reg_alpha": 1.9337364872373148,
  "reg_lambda": 0.0014437665473184575,
  "min_gain_to_split": 0.3424799228397572,
  "subsample": 0.7671021978724429,
  "colsample_bytree": 0.5592277215270216,
  "learning_rate": 0.016857309035168133,
  "n_estimators": 707,
  "weight_power": 0.45225216489327935,
  "tail_boost_a4": 1.0694607004564776

## 5. Re-evaluacion del mejor trial y politica post-hoc

Se recalcula el mejor trial para obtener probabilidades OOF completas y probar una politica post-hoc por multiplicadores. Esta politica no sustituye a la politica final; solo diagnostica si el problema de A4/A5 es de entrenamiento o de frontera de decision.

In [6]:
best_params, best_weight_params = params_from_trial(study.best_trial)
best_result = evaluate_config(best_params, best_weight_params, return_oof=True)
best_metrics = best_result["metrics"]
best_oof_proba = best_result["oof_proba"]
best_oof_pred = best_result["oof_pred"]

best_params_export = dict(best_params)
best_params_export["n_estimators"] = int(round(best_metrics["best_iteration_mean"]))

print(json.dumps({k: round(v, 6) for k, v in best_metrics.items() if k in ["score_compuesto", "macro_f1", "balanced_accuracy", "f1_a1", "f1_a2", "f1_a3", "f1_a4", "f1_a5", "infratriaje_critico_a1"]}, indent=2))

{
  "macro_f1": 0.570332,
  "balanced_accuracy": 0.588298,
  "f1_a1": 0.669389,
  "f1_a2": 0.659944,
  "f1_a3": 0.742032,
  "f1_a4": 0.497942,
  "f1_a5": 0.282353,
  "infratriaje_critico_a1": 0.018713,
  "score_compuesto": 0.514454
}


In [7]:
def apply_multipliers(proba: np.ndarray, multipliers: np.ndarray) -> np.ndarray:
    adjusted = proba * multipliers.reshape(1, -1)
    return np.argmax(adjusted, axis=1) + 1


def policy_objective(trial: optuna.Trial) -> float:
    multipliers = np.array(
        [
            trial.suggest_float("m1", 0.98, 1.02),
            trial.suggest_float("m2", 0.95, 1.10),
            trial.suggest_float("m3", 0.85, 1.05),
            trial.suggest_float("m4", 1.00, 1.50),
            trial.suggest_float("m5", 1.00, 3.00),
        ],
        dtype=float,
    )
    pred = apply_multipliers(best_oof_proba, multipliers)
    metrics = metricas_completas(y_train, pred)
    score = score_compuesto(metrics, baseline_metrics)
    for key, value in metrics.items():
        trial.set_user_attr(key, float(value))
    return score


policy_study = optuna.create_study(
    study_name=POLICY_STUDY_NAME,
    storage=POLICY_STORAGE_URL,
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE),
    load_if_exists=True,
)
policy_study.optimize(policy_objective, n_trials=POLICY_TRIALS, show_progress_bar=True)

best_policy_multipliers = np.array([policy_study.best_trial.params[f"m{i}"] for i in range(1, 6)], dtype=float)
policy_pred = apply_multipliers(best_oof_proba, best_policy_multipliers)
policy_metrics = metricas_completas(y_train, policy_pred)
policy_metrics["score_compuesto"] = score_compuesto(policy_metrics, baseline_metrics)

print("Best policy score:", policy_study.best_value)
print("Multipliers:", best_policy_multipliers.round(4).tolist())

[I 2026-06-07 09:35:09,196] A new study created in RDB with name: lgbm_bert_tail_policy_oof
Best trial: 0. Best value: 0.510064:   0%|          | 1/800 [00:00<06:29,  2.05it/s]

[I 2026-06-07 09:35:09,675] Trial 0 finished with value: 0.5100636462382853 and parameters: {'m1': 0.9949816047538945, 'm2': 1.0926071459614874, 'm3': 0.996398788362281, 'm4': 1.2993292420985183, 'm5': 1.312037280884873}. Best is trial 0 with value: 0.5100636462382853.


Best trial: 0. Best value: 0.510064:   0%|          | 2/800 [00:00<05:54,  2.25it/s]

[I 2026-06-07 09:35:10,099] Trial 1 finished with value: 0.4729456132425037 and parameters: {'m1': 0.9862397808134481, 'm2': 0.9587125418252299, 'm3': 1.023235229154987, 'm4': 1.3005575058716043, 'm5': 2.416145155592091}. Best is trial 0 with value: 0.5100636462382853.


Best trial: 2. Best value: 0.511712:   0%|          | 3/800 [00:01<05:32,  2.40it/s]

...

[I 2026-06-07 09:40:12,191] Trial 797 finished with value: 0.5146130574623825 and parameters: {'m1': 0.9979982854334685, 'm2': 1.0478523355408889, 'm3': 1.049770921094853, 'm4': 1.083068929985568, 'm5': 1.0701899443534024}. Best is trial 639 with value: 0.5158283638877699.


Best trial: 639. Best value: 0.515828: 100%|█████████▉| 799/800 [05:03<00:00,  2.54it/s]

[I 2026-06-07 09:40:12,590] Trial 798 finished with value: 0.5150059389567027 and parameters: {'m1': 0.9821110286366912, 'm2': 1.0449124298256691, 'm3': 1.0458918674705628, 'm4': 1.0761098645745273, 'm5': 1.0345082325330797}. Best is trial 639 with value: 0.5158283638877699.


Best trial: 639. Best value: 0.515828: 100%|██████████| 800/800 [05:03<00:00,  2.63it/s]


[I 2026-06-07 09:40:12,974] Trial 799 finished with value: 0.5151480433940462 and parameters: {'m1': 0.981097820493001, 'm2': 1.049994995449408, 'm3': 1.046285161176088, 'm4': 1.058239772259157, 'm5': 1.0002662149001567}. Best is trial 639 with value: 0.5158283638877699.
Best policy score: 0.5158283638877699
Multipliers: [0.9839, 1.031, 1.0332, 1.0925, 1.0009]


## 6. Comparacion, matrices y exportacion

Se exportan resultados separados. No se modifica el modelo congelado.

In [8]:
def row_from_metrics(modelo: str, metrics: dict[str, float]) -> dict[str, float | str]:
    keys = [
        "score_compuesto",
        "macro_f1",
        "weighted_f1",
        "balanced_accuracy",
        "f1_a1",
        "f1_a2",
        "f1_a3",
        "f1_a4",
        "f1_a5",
        "recall_a4",
        "precision_a4",
        "recall_a5",
        "precision_a5",
        "infratriaje_total",
        "sobretriaje_total",
        "infratriaje_critico_a1",
    ]
    row: dict[str, float | str] = {"modelo": modelo}
    for key in keys:
        row[key] = float(metrics.get(key, np.nan))
    return row


baseline_metrics_with_score = dict(baseline_metrics)
baseline_metrics_with_score["score_compuesto"] = baseline_score
comparison = pd.DataFrame(
    [
        row_from_metrics("lightgbm_final_bert_actual_oof", baseline_metrics_with_score),
        row_from_metrics("tail_tuned_training_argmax_oof", best_metrics),
        row_from_metrics("tail_tuned_policy_posthoc_oof", policy_metrics),
    ]
)
for col in ["score_compuesto", "macro_f1", "f1_a4", "f1_a5", "f1_a1", "f1_a2", "infratriaje_critico_a1"]:
    comparison[f"delta_{col}_vs_actual"] = comparison[col] - comparison.loc[0, col]

trials_path = OUT_DIR / "lgbm_bert_tail_tuning_trials.csv"
best_path = OUT_DIR / "lgbm_bert_tail_tuning_best_params.json"
comparison_path = OUT_DIR / "lgbm_bert_tail_tuning_oof_comparison.csv"
summary_path = OUT_DIR / "lgbm_bert_tail_tuning_summary.md"

study.trials_dataframe(attrs=("number", "value", "state", "params", "user_attrs", "duration")).to_csv(trials_path, index=False, encoding="utf-8")
comparison.to_csv(comparison_path, index=False, encoding="utf-8")

cm_baseline = confusion_matrix(y_train, baseline_oof["y_pred"], labels=CLASSES)
cm_best = confusion_matrix(y_train, best_oof_pred, labels=CLASSES)
cm_policy = confusion_matrix(y_train, policy_pred, labels=CLASSES)

adoption = (
    (best_metrics["macro_f1"] - baseline_metrics["macro_f1"] >= 0.003)
    and (best_metrics["f1_a4"] > baseline_metrics["f1_a4"])
    and ((best_metrics["f1_a5"] > baseline_metrics["f1_a5"]) or (best_metrics["recall_a5"] > baseline_metrics["recall_a5"]))
    and (best_metrics["f1_a1"] >= baseline_metrics["f1_a1"] - 0.005)
    and (best_metrics["f1_a2"] >= baseline_metrics["f1_a2"] - 0.005)
    and (best_metrics["infratriaje_critico_a1"] <= baseline_metrics["infratriaje_critico_a1"] + 0.001)
)

best_payload = {
    "study_name": STUDY_NAME,
    "policy_study_name": POLICY_STUDY_NAME,
    "baseline_metrics_oof": baseline_metrics,
    "baseline_score_compuesto": baseline_score,
    "best_trial_number": int(study.best_trial.number),
    "best_score_compuesto": float(study.best_value),
    "best_params_lgbm": best_params_export,
    "best_weight_params": best_weight_params,
    "best_metrics_argmax_oof": best_metrics,
    "best_policy_trial_number": int(policy_study.best_trial.number),
    "best_policy_multipliers": best_policy_multipliers.tolist(),
    "best_policy_metrics_oof": policy_metrics,
    "adoption_criteria_met_argmax": bool(adoption),
    "test_usage": "No se carga ni se usa test temporal en este notebook.",
}
best_path.write_text(json.dumps(best_payload, ensure_ascii=False, indent=2), encoding="utf-8")

display(comparison)
print(f"Trials -> {trials_path}")
print(f"Best params -> {best_path}")
print(f"Comparison -> {comparison_path}")

,modelo,score_compuesto,macro_f1,weighted_f1,balanced_accuracy,f1_a1,f1_a2,f1_a3,f1_a4,f1_a5,...,infratriaje_total,sobretriaje_total,infratriaje_critico_a1,delta_score_compuesto_vs_actual,delta_macro_f1_vs_actual,delta_f1_a4_vs_actual,delta_f1_a5_vs_actual,delta_f1_a1_vs_actual,delta_f1_a2_vs_actual,delta_infratriaje_critico_a1_vs_actual
0,lightgbm_final_bert_actual_oof,0.511160,0.567940,0.692585,0.586992,0.669482,0.662288,0.740868,0.500692,0.266368,...,0.170082,0.140908,0.018097,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
1,tail_tuned_training_argmax_oof,0.514454,0.570332,0.692279,0.588298,0.669389,0.659944,0.742032,0.497942,0.282353,...,0.171912,0.139049,0.018713,0.003294,0.002392,-0.002750,0.015985,-0.000092,-0.002345,0.000616
2,tail_tuned_policy_posthoc_oof,0.515828,0.571364,0.691226,0.588948,0.671576,0.660727,0.739312,0.498063,0.287140,...,0.175747,0.136803,0.019062,0.004669,0.003424,-0.002628,0.020771,0.002095,-0.001561,0.000966


Trials -> C:\Users\CARLOS\triaje-ia-tfg\reports\hyperparameter_tuning\lgbm_bert_tail_tuning_trials.csv
Best params -> C:\Users\CARLOS\triaje-ia-tfg\reports\hyperparameter_tuning\lgbm_bert_tail_tuning_best_params.json
Comparison -> C:\Users\CARLOS\triaje-ia-tfg\reports\hyperparameter_tuning\lgbm_bert_tail_tuning_oof_comparison.csv


In [ ]:
comparison_csv = comparison.to_csv(index=False, float_format="%.6f")
decision = (
    "El candidato cumple los criterios OOF de adopcion. Debe congelarse en artefactos nuevos y solo despues evaluarse una vez en test temporal."
    if adoption
    else "El candidato no cumple todos los criterios OOF de adopcion. Se documenta como experimento de sensibilidad y se mantiene el modelo congelado actual."
)

lines = [
    "# Tuning orientado a ESI 4/5",
    "",
    "Busqueda nocturna sobre LightGBM+BERT/SVD usando solo train y validacion OOF agrupada por paciente. El test temporal no se ha usado.",
    "",
    "## Comparacion OOF",
    "",
    "```csv",
    comparison_csv.strip(),
    "```",
    "",
    "## Mejor configuracion de entrenamiento",
    "",
    f"- Trial: {study.best_trial.number}",
    f"- Score compuesto: {study.best_value:.6f}",
    f"- Macro F1: {best_metrics['macro_f1']:.6f}",
    f"- F1 A4: {best_metrics['f1_a4']:.6f}",
    f"- F1 A5: {best_metrics['f1_a5']:.6f}",
    f"- F1 A1: {best_metrics['f1_a1']:.6f}",
    f"- F1 A2: {best_metrics['f1_a2']:.6f}",
    f"- Infratriaje critico A1: {best_metrics['infratriaje_critico_a1']:.6f}",
    "",
    "## Mejor politica post-hoc OOF",
    "",
    f"- Multiplicadores: {best_policy_multipliers.round(4).tolist()}",
    f"- Score compuesto: {policy_metrics['score_compuesto']:.6f}",
    f"- Macro F1: {policy_metrics['macro_f1']:.6f}",
    f"- F1 A4: {policy_metrics['f1_a4']:.6f}",
    f"- F1 A5: {policy_metrics['f1_a5']:.6f}",
    "",
    "## Decision metodologica",
    "",
    decision,
    "",
    "No se sobrescribe `models/lgbm_bert_final.joblib`. La politica post-hoc es diagnostica y no modifica la politica A1 final.",
    "",
    "## Artefactos",
    "",
    f"- Trials: `{trials_path.relative_to(PROJECT_ROOT)}`",
    f"- Parametros: `{best_path.relative_to(PROJECT_ROOT)}`",
    f"- Comparacion: `{comparison_path.relative_to(PROJECT_ROOT)}`",
    f"- Resumen: `{summary_path.relative_to(PROJECT_ROOT)}`",
    "",
    "## Matrices de confusion OOF",
    "",
    "### Baseline actual",
    "",
    pd.DataFrame(cm_baseline, index=CLASSES, columns=CLASSES).to_csv(),
    "",
    "### Mejor entrenamiento tail",
    "",
    pd.DataFrame(cm_best, index=CLASSES, columns=CLASSES).to_csv(),
    "",
    "### Mejor politica post-hoc",
    "",
    pd.DataFrame(cm_policy, index=CLASSES, columns=CLASSES).to_csv(),
]
summary_path.write_text("\n".join(lines), encoding="utf-8")
print(f"Resumen -> {summary_path}")

Resumen -> C:\Users\CARLOS\triaje-ia-tfg\reports\hyperparameter_tuning\lgbm_bert_tail_tuning_summary.md


: 

## 7. Cierre

Este notebook sirve para decidir si merece la pena congelar un nuevo candidato. Si no cumple los criterios OOF, el resultado sigue siendo util para la memoria como experimento de sensibilidad sobre clases minoritarias.